# Module 4: exercises  (solutions)

Use `students` and `rides` as described. Each check tells you what's wrong.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

## Exercise 1

**Load and inspect.** Load `students.csv` into `students` and store the number of **columns** in `n_cols` and the dtype of `exam_score`
(as a string, e.g. `"float64"`) in `score_dtype`.

In [ ]:
import numpy as np
import pandas as pd

students = pd.read_csv(DATA / "students.csv")
n_cols = students.shape[1]
score_dtype = str(students["exam_score"].dtype)

In [ ]:
assert n_cols == 7
assert score_dtype == "float64"
print("✅ Correct!")

## Exercise 2

**Filter.** Create `late_sleepers`: the rows of `students` where `sleep_hours` is **below 6**.
Then store in `late_mean` their mean `exam_score`.

In [ ]:
late_sleepers = students[students["sleep_hours"] < 6]
late_mean = late_sleepers["exam_score"].mean()

In [ ]:
assert (late_sleepers["sleep_hours"] < 6).all()
assert len(late_sleepers) == (students["sleep_hours"] < 6).sum()
assert abs(late_mean - late_sleepers["exam_score"].mean()) < 1e-9
print(f"✅ Correct! {len(late_sleepers)} students sleep under 6h, mean score {late_mean:.1f}")

## Exercise 3

**Top five.** Make `top5`: the 5 students with the highest `exam_score`, sorted from highest to lowest,
keeping only the columns `student_id`, `major` and `exam_score`.

In [ ]:
top5 = (students.sort_values("exam_score", ascending=False)
                .head(5)[["student_id", "major", "exam_score"]])

In [ ]:
assert list(top5.columns) == ["student_id", "major", "exam_score"]
assert len(top5) == 5
assert top5["exam_score"].is_monotonic_decreasing, "Sort from highest to lowest"
assert top5["exam_score"].iloc[0] == students["exam_score"].max()
print("✅ Correct!")

## Exercise 4

**New column.** Add to a copy of `students` called `df` a column `efficiency` = `exam_score / study_hours_week`
(points per study hour). Then find `best_major`, the major with the highest **median** efficiency.
(Careful: a student who reports 0 study hours would give `inf`. Check whether that happens: `df["efficiency"]` should be finite.)

In [ ]:
df = students.copy()
df["efficiency"] = df["exam_score"] / df["study_hours_week"]

best_major = df.groupby("major")["efficiency"].median().idxmax()

In [ ]:
assert "efficiency" in df.columns
assert np.isfinite(df["efficiency"]).all(), "Some study hours are 0: see the note in the prompt"
assert best_major in {"Science", "Arts", "Business", "Engineering"}
assert best_major == df.groupby("major")["efficiency"].median().idxmax()
print("✅ Correct! Best major by median efficiency:", best_major)

## Exercise 5

**Group comparison.** Does a part-time job go together with lower scores?
Build `job_summary`: a DataFrame indexed by `part_time_job` with columns `n` (count of students),
`mean_score` and `std_score`.

In [ ]:
job_summary = students.groupby("part_time_job").agg(
    n=("student_id", "count"),
    mean_score=("exam_score", "mean"),
    std_score=("exam_score", "std"),
)

In [ ]:
assert list(job_summary.columns) == ["n", "mean_score", "std_score"]
assert job_summary["n"].sum() == 200
assert set(job_summary.index) == {"yes", "no"}
print("✅ Correct!")
print(job_summary.round(2))

## Exercise 6

**Dates and weekdays.** Load `daily_rides.csv` into `rides`, convert `date` to datetime, then compute
`busiest_month`: the month number (1–12) with the highest **average** daily `rides`.

In [ ]:
rides = pd.read_csv(DATA / "daily_rides.csv")
rides["date"] = pd.to_datetime(rides["date"])
busiest_month = int(rides.groupby(rides["date"].dt.month)["rides"].mean().idxmax())

In [ ]:
assert pd.api.types.is_datetime64_any_dtype(rides["date"]), "Convert 'date' with pd.to_datetime"
assert busiest_month in range(1, 13)
assert 5 <= busiest_month <= 9, "Rides rise with temperature: expect a summer month"
print("✅ Correct! Busiest month:", busiest_month)

## Exercise 7

**Mean vs median.** Compute `rain_mean` and `rain_median` of the `rain_mm` column in `rides`, and set `skewed`
to `True` if the mean is **more than 1.5×** the median (a rough sign that a few extreme values pull the mean up),
else `False`. Many days have no rain, so think about what you expect.

In [ ]:
rain_mean = rides["rain_mm"].mean()
rain_median = rides["rain_mm"].median()
skewed = bool(rain_mean > 1.5 * rain_median)

In [ ]:
assert rain_mean == rides["rain_mm"].mean() and rain_median == rides["rain_mm"].median()
assert skewed is True, "Most days are dry, so the median is tiny compared with the mean"
print(f"✅ Correct! mean {rain_mean:.2f} mm vs median {rain_median:.2f} mm")